In [ ]:
import subprocess
import sys
from pathlib import Path

import torch

CODE = next(
    path.parent.parent
    for path in Path("/kaggle/input").rglob("harness.py")
    if path.parent.name == "octlm"
)
OUT = Path("/kaggle/working/day8")
MODEL = OUT / "qwen"
ADAPTER = OUT / "adapter-0.pt"
OUT.mkdir(parents=True, exist_ok=True)
if not torch.cuda.is_available() or torch.cuda.get_device_capability(0) != (7, 5):
    raise RuntimeError("Select a Kaggle T4 GPU")
print("Code:", CODE, "GPU:", torch.cuda.get_device_name(0), flush=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "tokenizers==0.22.2", "jinja2>=3.1.6,<4"],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "octlm.day6", "prepare", "--model", str(MODEL)], cwd=CODE, check=True
)
common = ["--model", str(MODEL), "--device", "cuda:0", "--adapter", str(ADAPTER)]
train = [sys.executable, "-m", "octlm.day8", "train", "--out", str(OUT / "train-0.jsonl")]
subprocess.run([*train, "--seed", "0", *common], cwd=CODE, check=True)
print((OUT / "train-0.jsonl").read_text().splitlines()[-1], flush=True)
out = OUT / "eval-0-int8.jsonl"
evaluate = [sys.executable, "-m", "octlm.day8", "eval", "--out", str(out), "--int8"]
subprocess.run([*evaluate, *common], cwd=CODE, check=True)
print(out.read_text().splitlines()[0], out.read_text().splitlines()[-1], sep="\n", flush=True)